In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: NVIDIA A40


In [2]:
!pip install -q transformers datasets accelerate sentencepiece huggingface_hub

In [3]:
from datasets import load_dataset

TRAIN_PATH = "/home/jovyan/work/hyde_train_split.jsonl"
VAL_PATH   = "/home/jovyan/work/hyde_val_split.jsonl"

dataset = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": VAL_PATH})
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 248525
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 27614
    })
})


In [4]:
# Load Model and Tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# MODEL_NAME = "google/flan-t5-small"
MODEL_NAME = "razent/SciFive-base-Pubmed_PMC"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model and Tokenizer loaded successfully!")

Loading razent/SciFive-base-Pubmed_PMC...


config.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Model and Tokenizer loaded successfully!


In [5]:
# Pre-processing (Tokenization)
# CRITICAL CHANGE: Increased target_length to 256 for paragraph generation
def tokenize_function(batch):
    # Inputs (Questions) are short
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=128,
        truncation=True
    )

    # Targets (Hypothetical Answers) are longer paragraphs
    labels = tokenizer(
        batch["target_text"],
        max_length=256,  # <--- Increased from 64 to 256
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

print("Data tokenized.")

Map:   0%|          | 0/248525 [00:00<?, ? examples/s]

Map:   0%|          | 0/27614 [00:00<?, ? examples/s]

Data tokenized.


In [6]:
# Training Setup
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./hyde_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,              # Slightly higher LR for generation usually helps T5
    per_device_train_batch_size=8,   # Lower batch size (8) because targets are longer (256 tokens)
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [7]:
# Train
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss
1,1.704300,1.610204
2,1.606400,1.559553
3,1.549800,1.539256


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=93198, training_loss=1.659648028424698, metrics={'train_runtime': 15489.6485, 'train_samples_per_second': 48.134, 'train_steps_per_second': 6.017, 'total_flos': 3.860305859837952e+16, 'train_loss': 1.659648028424698, 'epoch': 3.0})

In [8]:
# Test the Model (Hypothetical Answer Generation)
def generate_hypothetical_answer(question):
    input_text = "hypothetical answer: " + question
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,       # Allow it to generate a long paragraph
            num_beams=4,          # Beam search for better quality
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_q = "What are the different measurements and calculations performed during echocardiography?"
print(f"\nQuestion: {test_q}")
print(f"Generated HyDE Document:\n{generate_hypothetical_answer(test_q)}")


Question: What are the different measurements and calculations performed during echocardiography?
Generated HyDE Document:
During echocardiography, measurements such as left ventricular end-diastolic diameter (LVEDD), left atrial diameter (LAD), right ventricular ejection fraction (RVEF), and fractional shortening (FS) are performed. These measurements provide information about the structure and function of the heart.


In [9]:
# Save Model
# SAVE_DIR = "/content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/flan_t5_hyde_generator"
SAVE_DIR = "/home/jovyan/work/scifive_hyde_generator"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to: {SAVE_DIR}")

Model saved to: /home/jovyan/work/scifive_hyde_generator


In [11]:
from huggingface_hub import notebook_login
notebook_login()

In [2]:
from huggingface_hub import HfApi

api = HfApi()

repo_name = "hyde-sciFive-cardiology-generator-new"
username = "SandunR"  # replace this

api.create_repo(
    repo_id=f"{username}/{repo_name}",
    exist_ok=True,
    private=False
)

api.upload_folder(
    folder_path="/home/jovyan/work/scifive_hyde_generator",
    repo_id=f"{username}/{repo_name}",
    repo_type="model"
)

print("✅ Model uploaded successfully!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Model uploaded successfully!
